# 11 · Wrap-up and take-homes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/11-wrap-up-and-take-homes.ipynb)

*wrap-up · 5 min*

> 🇪🇸 **Cierre y ejercicios para casa** — Qué conecta los bloques 4, 5 y 6, más cinco ejercicios para casa.

What connects Blocks 4, 5 and 6, plus five take-home exercises.

## What you will be able to do

- State the one idea that connects the pseudoinverse, deconvolution and Tucker.
- Find the scaling trap in PCA on real, unstandardized data (take-home A).
- Build attention out of two contractions, and mask padded positions (take-home B).
- Compare a CP decomposition against the Tucker one you built (take-home C).
- Build correlated data from independent noise with Cholesky, and see why ignoring covariance understates portfolio risk (take-home D).
- Denoise a real voice recording by truncating the SVD of its STFT, and measure the result in SNR rather than by ear (take-home E).

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from scipy import signal

rng = np.random.default_rng(0)

## What you did today

> 🇪🇸 Lo que hiciste hoy.

1. **Section 01** — learned the vocabulary of tensors (axis, order, shape, slice,
   fiber, unfolding, contraction, decomposition), and that unfolding turns any
   tensor into a matrix without losing anything.
2. **Sections 02 and 05** — argued about what axes *mean*, and found that a batch
   axis and a time axis behave differently even when the shapes look identical.
3. **Sections 03 and 04** — indexed, broadcast, reshaped and transposed real
   tumour data and real medical images, and hit real problems: zero-variance
   pixels, and reshape silently destroying an image.
4. **Sections 06–10** — wrote contractions with `einsum`; solved an unsolvable
   20,433-equation system with the pseudoinverse; used recursion to forecast real
   airline traffic and to find an eigenvector; convolved and deconvolved a real
   photograph; and compressed a real taxi tensor 4.7× with Tucker, which found
   rush hour on its own.

### One idea connects sections 07, 09 and 10

**When a problem has no exact answer or no true inverse, you do not give up —
you find the best stable approximation.** The pseudoinverse does this for linear
systems, Richardson-Lucy for blurred images, and Tucker for tensors that are too
large to keep in full.

> 🇪🇸 Cuando un problema no tiene respuesta exacta ni inversa verdadera, no te
> rindes: buscas la mejor aproximación estable.

## Where to go next

- `torch.einsum` / `tf.einsum` / `jnp.einsum` — **identical syntax** to what you
  used today.
- [`tensorly`](https://tensorly.org) — proper Tucker and CP decompositions.
- `np.linalg` — the rest of Chapter 2: eigendecomposition, `lstsq`, `pinv`, `qr`,
  `cholesky`.
- `scipy.signal` and `skimage.restoration` — convolution and deconvolution
  beyond today.
- The five take-homes below.

### Optional: the same contraction in PyTorch

Everything today was NumPy, because that is what the workshop's real datasets
and verified numbers are built on. The einsum string does not change when you
move to a deep learning framework — only the array type does.

In [ ]:
# Optional. Colab has torch pre-installed; skip this cell if you prefer.
try:
    import torch
    photo = rng.standard_normal((8, 8, 3))
    w = np.array([0.2125, 0.7154, 0.0721])

    np_gray = np.einsum('hwc,c->hw', photo, w)
    pt_gray = torch.einsum('hwc,c->hw', torch.tensor(photo), torch.tensor(w))

    print(np.allclose(np_gray, pt_gray.numpy()))     # True — same string, same answer
except ImportError:
    print("torch not installed — nothing here you need")

---

## Take-home A — How many principal components are enough?

> 🇪🇸 Ejercicio para casa A: ¿cuántas componentes principales bastan?

**Real data contains a trap here. Find it.**

In [ ]:
bc = load_breast_cancer(); X, y = bc.data, bc.target

# TODO 1: Center X, run np.linalg.svd, and compute the fraction of variance each
#         component explains (variance is proportional to S**2).

# TODO 2: How many components explain 95% of the variance? The answer will look
#         TOO GOOD. Do not trust it yet.

# TODO 3: Print X.var(axis=0). The 30 measurements use different units — some are
#         areas in the thousands, some are ratios below 1. What is that doing?

# TODO 4: Redo everything on standardized data: (X - mean) / std. How many now?

# TODO 5: Scatter-plot the first 2 components, coloured by y. Do the two groups
#         separate?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
Xc = X - X.mean(axis=0)
S = np.linalg.svd(Xc, full_matrices=False)[1]
frac = S**2 / (S**2).sum()
n95 = np.argmax(np.cumsum(frac) >= 0.95) + 1      # 1  (!)
print(n95, round(frac[0], 3))                      # 1 0.982

print(np.sort(X.var(axis=0))[[0, -1]])             # ~0.0000075 up to ~324000

Xs = (X - X.mean(axis=0)) / X.std(axis=0)
S2 = np.linalg.svd(Xs, full_matrices=False)[1]
n95_scaled = np.argmax(np.cumsum(S2**2 / (S2**2).sum()) >= 0.95) + 1   # 10
print(n95_scaled)

# Without standardizing, the first component appears to explain 98.2% of the
# variance. IT IS AN ILLUSION: `worst area` has a variance around 323,000 while
# smoothness values sit below 1, so PCA reports the largest UNIT, not the
# largest PATTERN. After standardizing, the first component explains 44% and
# TEN components are needed.
#
# PCA KNOWS NOTHING ABOUT UNITS. Features on different scales must be
# standardized first.

# TODO 5:
# Z = Xs @ np.linalg.svd(Xs, full_matrices=False)[2][:2].T
# import matplotlib.pyplot as plt
# plt.scatter(Z[:, 0], Z[:, 1], c=y, s=8, cmap="coolwarm")

---

## Take-home B — Attention is two contractions

> 🇪🇸 Ejercicio para casa B: la atención son dos contracciones.

Attention is the mechanism that answers question 5 from section 05: *which parts
of a sequence matter most?* Protein language models use it so every amino acid
can look at every other one; recommenders use it to weight a user's past
interactions.

In [ ]:
np.random.seed(6)
batch, seq_len, dim = 4, 12, 16
Q, K, V = (np.random.randn(batch, seq_len, dim) for _ in range(3))

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

# TODO 1: With einsum, compute scores[b,i,j] = how much position i attends to
#         position j. Shape (4, 12, 12). Scale by 1/sqrt(dim).

# TODO 2: Apply softmax on the correct axis so each row of weights sums to 1.

# TODO 3: With einsum, combine V using those weights -> (4, 12, 16).

# TODO 4: Suppose the last 3 positions are padding, not real data. Build a mask,
#         set those scores to -np.inf BEFORE the softmax, and verify the padded
#         positions receive exactly zero weight.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
scores  = np.einsum('bid,bjd->bij', Q, K) / np.sqrt(dim)
weights = softmax(scores, axis=-1)
output  = np.einsum('bij,bjd->bid', weights, V)
print(scores.shape, weights.shape, output.shape)
print(np.allclose(weights.sum(axis=-1), 1.0))        # True

mask = np.zeros((seq_len, seq_len)); mask[:, -3:] = -np.inf
weights_masked = softmax(scores + mask, axis=-1)
print(weights_masked[..., -3:].max())                # 0.0 — exactly zero weight

# `scores` is Chapter 2's dot product (eq. 2.8); `output` is Chapter 2's linear
# combination (eq. 2.28). ATTENTION IS TWO CONTRACTIONS built from ideas you had
# already read.
#
# TODO 4 solves the variable-length problem from section 02: THE MASK IS HOW
# REAL MODELS HANDLE SEQUENCES AND VIDEOS OF DIFFERENT LENGTHS.

---

## Take-home C — CP decomposition, compared to Tucker

> 🇪🇸 Ejercicio para casa C: CP comparado con Tucker.

In [ ]:
# TODO 1: Build one rank-1 tensor with einsum from three random vectors of
#         length 4, 5 and 24. What shape is it? How many numbers define it?

# TODO 2: Compare that against 4*5*24. What is the compression of ONE rank-1 piece?

# TODO 3: pip install tensorly, run tensorly.decomposition.parafac on the taxi
#         tensor T with rank=3, and compare its error against your Tucker result.

# TODO 4: Which was more accurate at similar size? Why might that be?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
a, b, c = rng.standard_normal(4), rng.standard_normal(5), rng.standard_normal(24)
rank1 = np.einsum('i,j,k->ijk', a, b, c)     # (4, 5, 24) from only 33 numbers
print(rank1.shape, len(a) + len(b) + len(c), 4 * 5 * 24)   # (4,5,24) 33 480
print(round(480 / 33, 1))                                   # 14.5x for one piece

# TODO 3 — needs the taxi tensor from section 10:
# %pip install -q tensorly
# import tensorly as tl
# from tensorly.decomposition import parafac
# cp = parafac(tl.tensor(T), rank=3)
# err = tl.norm(tl.cp_to_tensor(cp) - T) / tl.norm(T)

# TUCKER IS USUALLY MORE ACCURATE AT EQUAL SIZE, because its dense core can
# represent interactions between components on different axes — something CP's
# strict sum of rank-1 pieces cannot do. CP is often preferred when
# interpretability matters, because each component is one simple pattern per
# axis.

---

## Take-home D — Cholesky: the factorization that builds

> 🇪🇸 Ejercicio para casa D: Cholesky, la factorización que construye.

Every factorization used today — LU, QR, eigendecomposition, SVD — takes an
existing object **apart**. Cholesky is the one exception: you use it to
**build**. Given a covariance matrix `Sigma` that is symmetric and
positive-definite, `np.linalg.cholesky` finds a lower-triangular `L` with
`L @ L.T == Sigma`. Feed `L` independent Gaussian noise and it hands back
correlated draws with *exactly* that covariance.

`Sigma[i, j]` is the **covariance** between asset `i` and asset `j` — how much
they move together, in the assets' own units. Its diagonal `Sigma[i, i]` is
each asset's own variance. **Correlation** (`corr`) is the same relationship
rescaled to sit between -1 and 1, so it is comparable between assets of
different volatility; `Sigma = outer(vol, vol) * corr` puts the original scale
back in.

If `z` is independent noise (`Cov(z) = I`) and `x = L @ z`, then
`Cov(x) = L Cov(z) L.T = L L.T = Sigma` — which is exactly why `L` turns
independent draws into correlated ones.

> 🇪🇸 `Sigma[i, j]` es la covarianza entre el activo `i` y el `j`: cuánto se
> mueven juntos. La diagonal es la varianza de cada activo. `corr` es la misma
> relación reescalada entre -1 y 1. Si `z` es ruido independiente
> (`Cov(z) = I`) y `x = L @ z`, entonces `Cov(x) = L Cov(z) L.T = L L.T =
> Sigma`: por eso `L` convierte ruido independiente en ruido correlacionado.

In [ ]:
vol = np.array([0.012, 0.015, 0.010])
corr = np.array([[1.00, 0.85, 0.20],
                  [0.85, 1.00, 0.20],
                  [0.20, 0.20, 1.00]])
Sigma = np.outer(vol, vol) * corr

weights = np.array([0.4, 0.4, 0.2])
mu = np.array([0.00030, 0.00035, 0.00020])
n_days, n_paths, initial_value = 252, 20_000, 100.0

rng = np.random.default_rng(5)
sample_sizes = [100, 1_000, 100_000]

# TODO 1: L = np.linalg.cholesky(Sigma). Verify np.allclose(L @ L.T, Sigma) is
#         True, and print L and the reconstruction L @ L.T, both rounded.

# TODO 2: For each n in sample_sizes, draw z = rng.standard_normal((3, n)),
#         build x = L @ z, and compute the Frobenius error between np.cov(x)
#         and Sigma. Confirm it shrinks as n grows. For the LARGEST n, also
#         print np.cov(z) (should look like the identity) and np.cov(x)
#         (should look like Sigma) — that is the whole trick, made visible.

# TODO 3: Simulate a CORRECT correlated portfolio. Draw
#         z_paths = rng.standard_normal((3, n_days * n_paths)), build
#         correlated_asset_returns = mu[:, None] + L @ z_paths, reshape to
#         (3, n_paths, n_days), combine with `weights` into one daily
#         portfolio return per path per day, and compound each path into
#         terminal_correlated = initial_value * prod(1 + daily_returns).

# TODO 4: Simulate the SAME portfolio again but WRONG: replace L with
#         independent_scale = np.diag(np.sqrt(np.diag(Sigma))) — same
#         individual volatilities, zero cross-asset correlation — and reuse
#         the SAME z_paths. Produce terminal_independent the same way.

# TODO 5: Plot terminal_correlated and terminal_independent as overlaid
#         histograms (density=True) on the same axes, labelled and legended.

# TODO 6: Compare std, and the 5th and 1st percentiles, of both. Which
#         distribution has the fatter left tail — and why, given that no
#         individual asset's volatility ever changed?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
L = np.linalg.cholesky(Sigma)
print(np.allclose(L @ L.T, Sigma))          # True
print(np.round(L, 4))
print(np.round(L @ L.T, 6))                 # matches Sigma

errors = []
for n in sample_sizes:
    z = rng.standard_normal((3, n))
    x = L @ z
    err = np.linalg.norm(np.cov(x) - Sigma)
    errors.append(err)
    print(n, err)
print(errors[0] > errors[1] > errors[2])    # True — error shrinks as n grows

print(np.round(np.cov(z), 3))               # close to the identity
print(np.round(np.cov(x), 6))               # close to Sigma
# Cov(x) = Cov(Lz) = L Cov(z) L.T ~ L I L.T = L L.T = Sigma. Independent noise
# in, correlated noise out — Cholesky is the "square root" that makes it work.

z_paths = rng.standard_normal((3, n_days * n_paths))

correlated_asset_returns = (mu[:, None] + L @ z_paths).reshape(3, n_paths, n_days)
portfolio_returns_correlated = np.einsum('a,apd->pd', weights, correlated_asset_returns)
terminal_correlated = initial_value * np.prod(1 + portfolio_returns_correlated, axis=1)

independent_scale = np.diag(np.sqrt(np.diag(Sigma)))
independent_asset_returns = (mu[:, None] + independent_scale @ z_paths).reshape(3, n_paths, n_days)
portfolio_returns_independent = np.einsum('a,apd->pd', weights, independent_asset_returns)
terminal_independent = initial_value * np.prod(1 + portfolio_returns_independent, axis=1)

import matplotlib.pyplot as plt
plt.hist(terminal_independent, bins=80, density=True, alpha=0.6,
         label="Assets simulated independently")
plt.hist(terminal_correlated, bins=80, density=True, alpha=0.6,
         label="Correct correlated simulation")
plt.xlabel("Terminal portfolio value")
plt.ylabel("Density")
plt.legend()
plt.show()

print(terminal_correlated.std(), terminal_independent.std())               # ~18.8  ~13.6
print(np.percentile(terminal_correlated, [1, 5]))                           # ~70.7 ~79.7
print(np.percentile(terminal_independent, [1, 5]))                          # ~79.9 ~86.9

# EVERY asset kept its own individual volatility in BOTH simulations —
# independent_scale used the SAME diagonal as Sigma. The only thing that
# changed is whether the simulation lets the three assets fall together.
# Ignoring the positive covariance did not touch any single asset's risk; it
# erased real cross-asset comovement and manufactured DIVERSIFICATION THAT
# ISN'T THERE — the correlated portfolio's distribution is wider and its
# lower tail is worse.
#
# This is NOT "correlation always increases risk." It is specific to THIS
# positively-correlated book: a negatively correlated pair would do the
# opposite, and ignoring it would UNDERSTATE diversification, not overstate
# it. What generalizes is only this: assuming independence when assets are
# not independent gets the TAILS of the distribution wrong.

### What the comparison shows

**Every individual asset kept the same volatility in both simulations.** The
only thing that changed is whether the simulation lets the three assets move
together. Ignoring the positive covariance did not touch any single asset's
risk; it erased real cross-asset comovement and manufactured diversification
that was never there — the correlated portfolio's terminal-value distribution
is wider, and its bad days are worse, than the (wrong) independent one.

**This is not "correlation always increases risk."** It is specific to this
book, where every pair is positively correlated. A negatively correlated pair
would do the opposite: ignoring it would make the simulation *understate*
diversification, not overstate it. What is general is only this: **assuming
independence when assets are not independent gets the tails of the
distribution wrong.**

> 🇪🇸 Cada activo conservó su propia volatilidad en ambas simulaciones — lo
> único que cambió es si la simulación permite que los tres se muevan juntos.
> Ignorar la covarianza positiva no tocó el riesgo individual: borró el
> comovimiento real y fabricó una diversificación que no existía. Esto **no**
> significa que "la correlación siempre aumenta el riesgo" — es específico de
> esta cartera, donde todo está correlacionado positivamente. Con correlación
> negativa ocurriría lo contrario. Lo único general es que **asumir
> independencia cuando los activos no lo son distorsiona las colas de la
> distribución.**

---

## Take-home E — Audio denoising by rank reduction

> 🇪🇸 Ejercicio para casa E: eliminar ruido de audio reduciendo el rango.

Section 10 used truncated SVDs of matrix unfoldings to build a Tucker
approximation of a real taxi tensor. This take-home applies the same
low-rank idea to the frequency × time matrix produced from sound.

**The recording is real**: a five-second CC0 voice sample by Bart Massey, from
[`pdx-cs-sound/wavs`](https://github.com/pdx-cs-sound/wavs), pinned to commit
`ed5ebcbbbc2d11f0adddc9b50b78d581c29f738c` so the file this notebook fetches
cannot silently change under you. It downloads at runtime and is checked
against a known SHA-256 — if the download is corrupted or does not match the
pinned file, `fetch_verified_wav` below raises instead of quietly handing you
something else. **The noise is not real** — it is added on purpose, with a
fixed seed and a target signal-to-noise ratio, precisely so there is a known
clean reference to measure against. Do not confuse the two: the recording is
real data, exactly like every other dataset today; the noise is the
controlled experiment.

### Why a waveform becomes a matrix

A recording is one axis: amplitude over time. The **short-time Fourier
transform** (STFT) slices it into overlapping windows and Fourier-transforms
each one, producing a matrix `Z` with two axes — **frequency × time**. Row `i`
is "how much of frequency `f_i` is present"; column `j` is "during time window
`t_j`." Nothing earlier today paired frequency against time this way.

Because `Z` is a matrix, the SVD from sections 07 and 10 applies unchanged —
except `Z` is **complex**, and truncating its SVD keeps both magnitude and
phase. Reconstructing from magnitude alone would throw phase away and produce
audible distortion, so the truncated matrix goes straight into the inverse
STFT.

Speech energy concentrates in a handful of dominant frequency-time patterns —
a few singular vectors carry most of the signal. Broadband, unstructured noise
has no such structure: it tends to spread its energy across many singular
directions, including many smaller ones. Keeping only the largest `k`
singular values keeps most of the speech and discards a disproportionate
share of the noise.

> 🇪🇸 La STFT convierte una onda de una dimensión (amplitud en el tiempo) en
> una matriz de dos ejes: frecuencia × tiempo. La voz concentra su energía en
> pocas direcciones singulares dominantes; el ruido de banda ancha tiende a
> repartir su energía entre muchas direcciones singulares, incluidas muchas
> pequeñas. Por eso conservar solo las `k` mayores retiene la voz y descarta
> una parte desproporcionada del ruido — pero **esto no es un eliminador de
> ruido universal**: la comprobación real es el SNR medido, no cómo suena.

**This is not a universal denoiser.** It only works to the extent that the
noise really is broadband relative to a structured signal — narrowband noise,
or noise correlated with the signal, is not separated this way. The proof
either way is the measured SNR below, not how it sounds.

In [ ]:
VOICE_URL = "https://raw.githubusercontent.com/pdx-cs-sound/wavs/ed5ebcbbbc2d11f0adddc9b50b78d581c29f738c/voice.wav"
VOICE_SHA256 = "2c4b4d9d5f90715fdbf599869a465d521638f40ca978b186df96f1543a4d67dc"

def fetch_verified_wav(url, expected_sha256):
    """Download a WAV and refuse to proceed if it does not match the pinned
    checksum. No silent fallback to synthetic data on failure."""
    import hashlib
    import io
    import urllib.request
    from scipy.io import wavfile
    raw = urllib.request.urlopen(url, timeout=30).read()
    got = hashlib.sha256(raw).hexdigest()
    if got != expected_sha256:
        raise ValueError(
            f"checksum mismatch for {url}: expected {expected_sha256}, got "
            f"{got}. Refusing to use unverified audio data.")
    return wavfile.read(io.BytesIO(raw))

def snr_db(reference, estimate):
    """Energy-based SNR in dB. `reference` is always the real clean signal."""
    return 10 * np.log10(np.sum(reference**2) / np.sum((estimate - reference)**2))

# TODO 1: fs, clean_i16 = fetch_verified_wav(VOICE_URL, VOICE_SHA256).
#         Convert to float in [-1, 1] (divide by 32768), and average channels
#         to mono if clean.ndim > 1. Print fs, duration in seconds, and shape.

# TODO 2: With rng = np.random.default_rng(42) and TARGET_SNR_DB = 5.0, build
#         additive noise scaled from the CLEAN SIGNAL'S OWN MEAN POWER (not an
#         arbitrary standard deviation) so that clean + noise lands at the
#         target SNR. Verify with snr_db(clean, noisy).

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
fs, clean_i16 = fetch_verified_wav(VOICE_URL, VOICE_SHA256)
clean = clean_i16.astype(np.float64) / 32768.0
if clean.ndim > 1:
    clean = clean.mean(axis=1)
print(fs, round(len(clean) / fs, 3), clean.shape)      # 48000 4.949 (237568,)

rng = np.random.default_rng(42)
TARGET_SNR_DB = 5.0
noise = rng.standard_normal(clean.shape)
scale = np.sqrt(np.mean(clean**2) / (np.mean(noise**2) * 10**(TARGET_SNR_DB / 10)))
noisy = clean + scale * noise
print(round(snr_db(clean, noisy), 2))                   # 5.0 -- exactly the target, by construction

# fetch_verified_wav is not decorative: it raises ValueError instead of
# silently returning something else if the download is corrupted or does not
# match the pinned file. voice.wav ITSELF is real -- a five-second CC0
# recording. The noise added here is the controlled, synthetic part of the
# experiment: it exists only so `clean` is a known reference an SNR can be
# measured against.

In [ ]:
# TODO 3: f, t, Z = signal.stft(noisy, fs=fs, nperseg=1024, noverlap=512).
#         Z is COMPLEX -- frequency bins x time frames. Print Z.shape and the
#         full possible rank, min(Z.shape).

# TODO 4: U, s, Vh = np.linalg.svd(Z, full_matrices=False), on the COMPLEX
#         matrix directly so phase survives truncation, not magnitude alone.
#         For k in [2, 5, 10, 20, 40, 80, len(s)]: build
#         Z_k = (U[:, :k] * s[:k]) @ Vh[:k, :], run
#         signal.istft(Z_k, fs=fs, nperseg=1024, noverlap=512), align its
#         length to `clean`, and print k, the retained singular-value energy
#         sum(s[:k]**2) / sum(s**2), and snr_db(clean, reconstruction).

# TODO 5: Pick the k with the best SNR among the candidates above. Report its
#         retained energy, its SNR, and the improvement over the noisy SNR
#         from TODO 2.

# TODO 6: Build ONE common peak-scale factor from
#         max(|noisy|, |denoised|, |clean|) and make playback-only copies
#         scaled by it -- SNR itself is computed on the unscaled signals
#         above, never on these copies. Then display Audio players for the
#         noisy ("before") and denoised ("after") copies. You may run this
#         cell to listen, but do not save its Audio output into the tracked
#         notebook: Audio() output contains embedded base64 data and must
#         not be committed.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
f, t, Z = signal.stft(noisy, fs=fs, nperseg=1024, noverlap=512)
full_rank = min(Z.shape)
print(Z.shape, full_rank)                               # (513, 465) 465

U, s, Vh = np.linalg.svd(Z, full_matrices=False)
for k in [2, 5, 10, 20, 40, 80, len(s)]:
    Zk = (U[:, :k] * s[:k]) @ Vh[:k, :]
    _, x_rec = signal.istft(Zk, fs=fs, nperseg=1024, noverlap=512)
    n = min(len(x_rec), len(clean))
    energy = np.sum(s[:k]**2) / np.sum(s**2)
    print(k, round(energy * 100, 1), round(snr_db(clean[:n], x_rec[:n]), 2))
# k    energy%  SNR dB
# 2     33.2     2.29
# 5     51.1     4.76
# 10    61.9     6.78
# 20    70.1     8.48
# 40    78.3     9.08   <- best of these candidates
# 80    87.1     7.68   <- WORSE than k=40: noise has leaked back in
# 465  100.0     5.00   <- full rank matches `noisy` to numerical precision

k = 40
Zk = (U[:, :k] * s[:k]) @ Vh[:k, :]
_, x_rec = signal.istft(Zk, fs=fs, nperseg=1024, noverlap=512)
n = min(len(x_rec), len(clean))
denoised, clean_a, noisy_a = x_rec[:n], clean[:n], noisy[:n]

snr_before = snr_db(clean_a, noisy_a)
snr_after = snr_db(clean_a, denoised)
print(round(snr_before, 2), round(snr_after, 2), round(snr_after - snr_before, 2))
# 5.0 9.08 4.08

peak = max(np.abs(clean_a).max(), np.abs(noisy_a).max(), np.abs(denoised).max())
noisy_play = noisy_a / peak
denoised_play = denoised / peak

from IPython.display import Audio, display
display(Audio(noisy_play, rate=fs))       # "before"
display(Audio(denoised_play, rate=fs))    # "after"

# k=40 keeps 40 of 465 possible components -- 8.6% of full rank -- and
# recovers 4.08 dB of SNR: real, but modest, not a miracle. k=2 and k=5 keep
# too little of the SPEECH itself to beat the noisy baseline by much. k=80
# already lets enough noise back into smaller-but-still-significant singular
# directions that SNR gets WORSE than at k=40 -- more components is not
# always better. At the full rank of 465 the reconstruction matches `noisy`
# to numerical precision: proof that whatever denoising happened at k=40
# came specifically from truncating, not from the STFT -> SVD -> ISTFT round
# trip itself.

### What the numbers say

Keeping 40 of 465 possible singular directions (8.6% of full rank, 78.3% of
the singular-value energy) raised the SNR from 5.00 dB to 9.08 dB — a real
**+4.08 dB** improvement, not a dramatic one. Fewer components (`k=2`, `k=5`)
discard too much of the speech itself; more (`k=80`) already lets noise back
in, and SNR gets worse again. At the full rank the reconstruction matches the
noisy signal to numerical precision, which is the honest control: the
denoising is entirely a property of truncating, not of the STFT/SVD/ISTFT
machinery itself.

**Do not generalize this to "truncated SVD removes noise."** It suppresses
noise that is broadband and unstructured relative to a signal that
concentrates in a few dominant directions — the same low-rank argument
section 10 used on the taxi tensor, applied here to sound instead of trip
counts. Narrowband noise, or noise correlated with the speech itself, would
not separate out this way, and the only way to know which situation you are
in is to measure the SNR, the way this take-home just did.

> 🇪🇸 Conservar 40 de 465 direcciones singulares posibles (8.6% del rango
> completo, 78.3% de la energía de los valores singulares) subió el SNR de
> 5.00 dB a 9.08 dB — una mejora real de **+4.08 dB**, no espectacular. Menos
> componentes descartan demasiada voz; más vuelven a dejar entrar ruido y el
> SNR empeora. En el rango completo la reconstrucción coincide con la señal
> ruidosa hasta la precisión numérica, lo cual es el control honesto: la
> reducción de ruido es una propiedad de truncar, no del mecanismo
> STFT/SVD/ISTFT en sí. **No generalices esto a "la SVD truncada siempre
> elimina el ruido."** Solo funciona cuando el ruido es de banda ancha y no
> estructurado frente a una señal que se concentra en pocas direcciones
> dominantes — el mismo argumento de bajo rango que la sección 10 usó con el
> tensor de taxis, aplicado aquí al sonido. La única forma de saberlo es
> medir el SNR, como se acaba de hacer.

## Thank you

> 🇪🇸 Gracias por venir. Pregunta en Discord en español o en inglés — lo que te
> permita preguntar más rápido.

Questions stay welcome in Discord, in Spanish or English. The
[handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)
has everything from today, including the facilitator notes.

---

## Done with this section

That is the whole workshop. Thank you for coming.

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)